# Table 1

In [1]:
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import norm, pearsonr

## Filenames

In [2]:
start = 1940
stop = 2023
nens = 10

namelist_f = [
    [
        f"/glade/work/smhenry/NeuralGCM/data/tracks/factual/ens{X}_{YYYY}_JASO_TC_tracks_factual.txt"
        for X in range(1, nens + 1)
    ]
    for YYYY in range(start, stop + 1)
]

## Counting functions

In [3]:
def count_NA(file):
    """
    counts the number of TCs from TempestExtremes output in the North Atlantic Basin
    input: file, a file name which is an output from TempestExtremes
    output: count, the number of TCs tracked in the file
    """
    count = 0
    in_TC = False
    is_in_NA = False

    with open(file, "r") as df:
        for line in df:
            if line.startswith("start"):
                if is_in_NA:
                    count += 1
                is_in_NA = False
                in_TC = True
            elif in_TC:
                data = line.split()
                if len(data) >= 4:
                    lon = float(data[2])
                    lat = float(data[3])

                    if (
                        (285 <= lon <= 360 and 0 <= lat <= 50)
                        or (276 <= lon < 285 and 10 <= lat <= 50)
                        or (262 <= lon < 276 and 16.5 <= lat <= 50)
                    ):
                        is_in_NA = True

        if is_in_NA:
            count += 1

    return count


def count_NWP(file):
    """
    counts the number of TCs from TempestExtremes output in the Northwest Pacific Basin
    input: file, a file name which is an output from TempestExtremes
    output: count, the number of TCs tracked in the file
    """
    count = 0
    in_TC = False
    is_in_NWP = False

    with open(file, "r") as df:
        for line in df:
            if line.startswith("start"):
                if is_in_NWP:
                    count += 1
                is_in_NWP = False
                in_TC = True
            elif in_TC:
                data = line.split()
                if len(data) >= 4:
                    lon = float(data[2])
                    lat = float(data[3])

                    if 100 <= lon <= 180 and 0 <= lat <= 50:  # need to update
                        is_in_NWP = True

        if is_in_NWP:
            count += 1

    return count


def count_NEP(file):
    """
    counts the number of TCs from TempestExtremes output in the Northeast Pacific Basin
    input: file, a file name which is an output from TempestExtremes
    output: count, the number of TCs tracked in the file
    """
    count = 0
    in_TC = False
    is_in_NEP = False

    with open(file, "r") as df:
        for line in df:
            if line.startswith("start"):
                if is_in_NEP:
                    count += 1
                is_in_NEP = False
                in_TC = True
            elif in_TC:
                data = line.split()
                if len(data) >= 4:
                    lon = float(data[2])
                    lat = float(data[3])

                    if (
                        (275 <= lon <= 284 and 0 <= lat <= 10)
                        or (260 <= lon < 275 and 0 <= lat <= 16.5)
                        or (180 <= lon < 260 and 0 <= lat <= 50)
                    ):
                        is_in_NEP = True

        if is_in_NEP:
            count += 1

    return count


def count_TCs_ibtracs(year, basin, type):  # working yay!
    """
    returns the count of TCs from IBTrACS in a given basin for a given year
    input:
        year (int): desired year
        basin (str): desired basin
            keys: "AL", "WP", "EP"
        type (str): storm type
            keys: "TD" (tropical depression), "TS" (tropical storm), "HR" or "TY" (hurricane or typhoon, basin dependent)
            also see (DB: disturbance, TD: tropical depression, TS: tropical storm, TY: typhoon, ST: super typhoon, TC: tropical cyclone,
                    HU,HR: hurricane, SD: subtropical depression, SS: subtropical storm, EX: extratropical systems, PT: post tropical,
                    IN: inland, DS: dissipating, LO: low, WV: tropical wave, ET: extrapolated, MD: monsoon depression, XX: unknown)
    output:
        count (int): count of TCS
    criteria:
        TCs must last at least 54 hours to be consistent with TempestExtremes tracking
    """

    # open dataframe
    df = pd.read_csv(
        f"/glade/work/smhenry/NeuralGCM/data/ibtracs/IBTrACS_{year}_JASO.csv"
    )
    df["time"] = pd.to_datetime(df["time"])

    # Calculate storm durations
    storm_durations = df.groupby("stormid")["time"].agg(
        lambda x: (x.max() - x.min()).total_seconds() / 3600
    )

    # Keep only storms lasting at least 54 hours
    valid_storms = storm_durations[storm_durations >= 54].index
    df = df[df["stormid"].isin(valid_storms) & df["stormid"].str.contains(basin)]

    # Filter by storm type
    type_dict = {
        "TD": "HU|HR|TY|ST|TS|TC|SS|TD|SD",
        "TS": "HU|HR|TY|ST|TS|TC|SS",
        "HU": "HU|HR|TY|ST",
        "TY": "HU|HR|TY|ST",
    }

    if type in type_dict:
        df = df[df["type"].str.contains(type_dict[type])]

    return df["stormid"].nunique()

In [4]:
def count_NA_mo(file, mo):
    """
    counts the number of TCs from TempestExtremes output in the North Atlantic Basin
    input: file, a file name which is an output from TempestExtremes
    output: count, the number of TCs tracked in the file
    """
    count = 0
    in_TC = False
    is_in_NA = False
    in_mo = False

    with open(file, "r") as df:
        for line in df:
            if line.startswith("start"):
                if (
                    is_in_NA and in_mo
                ):  # this addresses the last TC, then bools are reset
                    count += 1
                is_in_NA = False
                in_TC = True
                if line.split()[3] == mo:
                    in_mo = True
                else:
                    in_mo = False
            elif in_TC:
                data = line.split()
                if len(data) >= 4:
                    lon = float(data[2])
                    lat = float(data[3])

                    if (
                        (285 <= lon <= 360 and 0 <= lat <= 50)
                        or (276 <= lon < 285 and 10 <= lat <= 50)
                        or (262 <= lon < 276 and 16.5 <= lat <= 50)
                    ):
                        is_in_NA = True

        if is_in_NA and in_mo:
            count += 1

    return count


def count_NWP_mo(file, mo):
    """
    counts the number of TCs from TempestExtremes output in the Northwest Pacific Basin
    input: file, a file name which is an output from TempestExtremes
    output: count, the number of TCs tracked in the file
    """
    count = 0
    in_TC = False
    is_in_NWP = False
    in_mo = None

    with open(file, "r") as df:
        for line in df:
            if line.startswith("start"):
                if (
                    is_in_NWP and in_mo
                ):  # this addresses the last TC, then bools are reset
                    count += 1
                is_in_NWP = False
                in_TC = True
                if line.split()[3] == mo:
                    in_mo = True
                else:
                    in_mo = False
            elif in_TC:
                data = line.split()
                if len(data) >= 4:
                    lon = float(data[2])
                    lat = float(data[3])

                    if 100 <= lon <= 180 and 0 <= lat <= 50:
                        is_in_NWP = True

        if is_in_NWP and in_mo:
            count += 1

    return count


def count_NEP_mo(file, mo):
    """
    counts the number of TCs from TempestExtremes output in the Northeast Pacific Basin
    input: file, a file name which is an output from TempestExtremes
    output: count, the number of TCs tracked in the file
    """
    count = 0
    in_TC = False
    is_in_NEP = False
    in_mo = None

    with open(file, "r") as df:
        for line in df:
            if line.startswith("start"):
                if (
                    is_in_NEP and in_mo
                ):  # this addresses the last TC, then bools are reset
                    count += 1
                is_in_NEP = False
                in_TC = True
                if line.split()[3] == mo:
                    in_mo = True
                else:
                    in_mo = False
            elif in_TC:
                data = line.split()
                if len(data) >= 4:
                    lon = float(data[2])
                    lat = float(data[3])

                    if (
                        (275 <= lon <= 284 and 0 <= lat <= 10)
                        or (260 <= lon < 275 and 0 <= lat <= 16.5)
                        or (180 <= lon < 260 and 0 <= lat <= 50)
                    ):
                        is_in_NEP = True

        if is_in_NEP and in_mo:
            count += 1

    return count


def count_TCs_ibtracs_mo(
    year, mo, basin, type
):  # updated to filter by storm start month
    """
    Returns the count of TCs from IBTrACS in a given basin for a specific year and month (based on storm start month).

    Inputs:
        year (int): desired year
        mo (int): desired month (1–12)
        basin (str): desired basin
            keys: "AL", "WP", "EP"
        type (str): storm type
            keys: "TD" (tropical depression), "TS" (tropical storm), "HR" or "TY" (hurricane or typhoon)
    Output:
        count (int): count of valid tropical cyclones (TCs)

    Criteria:
        - Storms must last at least 54 hours
        - Storm must start in the given month
    """
    # Open dataframe
    df = pd.read_csv(
        f"/glade/work/smhenry/NeuralGCM/data/ibtracs/IBTrACS_{year}_JASO.csv"
    )
    df["time"] = pd.to_datetime(df["time"])

    # Calculate storm durations
    storm_durations = df.groupby("stormid")["time"].agg(
        lambda x: (x.max() - x.min()).total_seconds() / 3600
    )

    # Keep only storms lasting at least 54 hours
    valid_storms = storm_durations[storm_durations >= 54].index
    df = df[df["stormid"].isin(valid_storms) & df["stormid"].str.contains(basin)]

    # Filter by storm type
    type_dict = {
        "TD": "HU|HR|TY|ST|TS|TC|SS|TD|SD",
        "TS": "HU|HR|TY|ST|TS|TC|SS",
        "HU": "HU|HR|TY|ST",
        "TY": "HU|HR|TY|ST",
    }

    if type in type_dict:
        df = df[df["type"].str.contains(type_dict[type])]

    # Get the first timestamp per storm
    storm_start_month = df.groupby("stormid")["time"].min().dt.month
    storms_starting_in_month = storm_start_month[storm_start_month == mo].index

    df = df[df["stormid"].isin(storms_starting_in_month)]

    return df["stormid"].nunique()

## Processing functions

In [5]:
def process_counts(namelist, basin, start, exclude=None, ens=False, min_max=False):
    """
    inputs:
        namelist: list of strs of filenames
        basin: str, "NA", "NWP", "NEP"
        start: start year of simulation data analysis
        exclude: list of sets of [[yr, ens]] to exclude
        ens: bool, contains ens data or not
        min_max: bool, return min max data for each year or not

    returns:
        ensmean_count: list of yearly ensemble mean TC counts, or if ens=False, just the yearly counts
        twenty: 20th percentile for each year, None if ens=False
        eighty: 80th percentile for each year, None if ens=False
        min: yearly min TC count, None if ens=False
        max: yearly max TC count, None if ens=False
    """

    nyr = np.shape(namelist)[0]
    if ens:
        nens = np.shape(namelist)[1]
    else:
        nens = 1

    # initalize array of TC counts for storage
    if ens:
        counts = np.zeros((nyr, nens))
    else:
        counts = np.zeros((nyr, 1))

    # count
    for iyr in range(nyr):
        if ens:
            for iens in range(nens):
                if exclude:
                    for i in range(len(exclude)):
                        if iyr == (exclude[i][0] - start) and iens == (
                            exclude[i][1] - 1
                        ):
                            counts[iyr][iens] = np.nan

                        else:
                            if basin == "NA":
                                counts[iyr][iens] = count_NA(namelist[iyr][iens])
                            elif basin == "NWP":
                                counts[iyr][iens] = count_NWP(namelist[iyr][iens])
                            elif basin == "NEP":
                                counts[iyr][iens] = count_NEP(namelist[iyr][iens])
                            else:
                                counts[iyr][iens] = np.nan

                else:
                    if basin == "NA":
                        counts[iyr][iens] = count_NA(namelist[iyr][iens])
                    elif basin == "NWP":
                        counts[iyr][iens] = count_NWP(namelist[iyr][iens])
                    elif basin == "NEP":
                        counts[iyr][iens] = count_NEP(namelist[iyr][iens])
                    else:
                        counts[iyr][iens] = np.nan

        else:
            if basin == "NA":
                counts[iyr] = count_NA(namelist[iyr])
            elif basin == "NWP":
                counts[iyr] = count_NWP(namelist[iyr])
            elif basin == "NEP":
                counts[iyr] = count_NEP(namelist[iyr])
            else:
                counts[iyr] = np.nan

    # take the ensemble mean and get a 20th and 80th percentile spread

    if ens:
        # initalize arrays for storage
        count = np.zeros((nyr, 1))
        twenty = np.zeros((nyr, 1))
        eighty = np.zeros((nyr, 1))
        min = np.zeros((nyr, 1))
        max = np.zeros((nyr, 1))

        for iyr in range(nyr):
            # count for that year
            yr_count = counts[iyr][:]

            # take mean across year
            count[iyr] = np.nanmean(yr_count)

            # take the 20th and 80th percentiles
            twenty[iyr] = np.nanpercentile(yr_count, 20)
            eighty[iyr] = np.nanpercentile(yr_count, 80)

            # take the mins and maxs
            if min_max:
                min[iyr] = np.nanmin(yr_count)
                max[iyr] = np.nanmax(yr_count)
            else:
                min = None
                max = None

    else:
        # initalize arrays for storage
        count = np.zeros((nyr, 1))
        twenty = np.zeros((nyr, 1))
        eighty = np.zeros((nyr, 1))
        min = np.zeros((nyr, 1))
        max = np.zeros((nyr, 1))

        for iyr in range(nyr):
            # count for that year
            yr_count = counts[iyr][:]

            # take mean across year
            count[iyr] = np.nanmean(yr_count)

            # take the 20th and 80th percentiles
            twenty[iyr] = np.nanpercentile(yr_count, 20)
            eighty[iyr] = np.nanpercentile(yr_count, 80)

            # take the mins and maxs
            if min_max:
                min[iyr] = np.nanmin(yr_count)
                max[iyr] = np.nanmax(yr_count)
            else:
                min = None
                max = None

    return count.flatten(), twenty, eighty, min, max

## Process data

In [6]:
f_exclude = [
    [1992, 3],
    [2017, 3],
]

# north atlantic
f_ensmean_count_NA, f_twenty_NA, f_eighty_NA, f_min_NA, f_max_NA = process_counts(
    namelist_f, "NA", start, ens=True, min_max=True, exclude=f_exclude
)
# northwest pacific
f_ensmean_count_NWP, f_twenty_NWP, f_eighty_NWP, f_min_NWP, f_max_NWP = process_counts(
    namelist_f, "NWP", start, ens=True, min_max=True, exclude=f_exclude
)
# northeast pacific
f_ensmean_count_NEP, f_twenty_NEP, f_eighty_NEP, f_min_NEP, f_max_NEP = process_counts(
    namelist_f, "NEP", start, ens=True, min_max=True, exclude=f_exclude
)

In [7]:
obs_TC_count_NA = []
obs_HR_count_NA = []
obs_TC_count_NEP = []
obs_TY_count_NEP = []
obs_TC_count_NWP = []
obs_TY_count_NWP = []

for year in range(start, stop + 1):
    if year < 1949:
        if year < 1945:
            obs_TC_count_NWP.append(np.nan)
            obs_TY_count_NWP.append(np.nan)
        else:
            obs_TC_count_NWP.append(count_TCs_ibtracs(year, "WP", "TD"))
            obs_TY_count_NWP.append(count_TCs_ibtracs(year, "WP", "TY"))

        obs_TC_count_NEP.append(np.nan)
        obs_TY_count_NEP.append(np.nan)
    else:
        obs_TC_count_NEP.append(count_TCs_ibtracs(year, "EP", "TD"))
        obs_TY_count_NEP.append(count_TCs_ibtracs(year, "EP", "TY"))
        obs_TC_count_NWP.append(count_TCs_ibtracs(year, "WP", "TD"))
        obs_TY_count_NWP.append(count_TCs_ibtracs(year, "WP", "TY"))

    obs_TC_count_NA.append(count_TCs_ibtracs(year, "AL", "TD"))
    obs_HR_count_NA.append(count_TCs_ibtracs(year, "AL", "HU"))

In [18]:
yr_list = np.arange(start, stop + 1, 1)

i_1940 = np.where(yr_list == 1940)[0][0]
i_1945 = np.where(yr_list == 1945)[0][0]
i_1949 = np.where(yr_list == 1949)[0][0]
i_1970 = np.where(yr_list == 1970)[0][0]
i_1985 = np.where(yr_list == 1985)[0][0]
i_1987 = np.where(yr_list == 1987)[0][0]

# corr list: (1940-2023,1949-2023,1949-1985,1987-2023,1970-2023)
corr_yr_list = [[1940, 2023], [1949, 2023], [1949, 1985], [1987, 2023], [1970, 2023]]

corr_TC_NA_list = np.empty((len(corr_yr_list)))
corr_HR_NA_list = np.empty((len(corr_yr_list)))

corr_TC_NWP_list = np.empty((len(corr_yr_list)))
corr_HR_NWP_list = np.empty((len(corr_yr_list)))

corr_TC_NEP_list = np.empty((len(corr_yr_list)))
corr_HR_NEP_list = np.empty((len(corr_yr_list)))

p_TC_NA_list = np.empty((len(corr_yr_list)))
p_HR_NA_list = np.empty((len(corr_yr_list)))

p_TC_NWP_list = np.empty((len(corr_yr_list)))
p_HR_NWP_list = np.empty((len(corr_yr_list)))

p_TC_NEP_list = np.empty((len(corr_yr_list)))
p_HR_NEP_list = np.empty((len(corr_yr_list)))

for i, corr_yrs in enumerate(corr_yr_list):
    i_yr1 = np.where(yr_list == corr_yrs[0])[0][0]
    i_yr2 = np.where(yr_list == corr_yrs[1])[0][0]

    corr_TC_NA_list[i], p_TC_NA_list[i] = pearsonr(
        f_ensmean_count_NA[i_yr1 : i_yr2 + 1], obs_TC_count_NA[i_yr1 : i_yr2 + 1]
    )

    corr_HR_NA_list[i], p_HR_NA_list[i] = pearsonr(
        f_ensmean_count_NA[i_yr1 : i_yr2 + 1], obs_HR_count_NA[i_yr1 : i_yr2 + 1]
    )

    corr_TC_NEP_list[i], p_TC_NEP_list[i] = pearsonr(
        f_ensmean_count_NEP[i_yr1 : i_yr2 + 1], obs_TC_count_NEP[i_yr1 : i_yr2 + 1]
    )
    corr_HR_NEP_list[i], p_HR_NEP_list[i] = pearsonr(
        f_ensmean_count_NEP[i_yr1 : i_yr2 + 1], obs_TY_count_NEP[i_yr1 : i_yr2 + 1]
    )

    corr_TC_NWP_list[i], p_TC_NWP_list[i] = pearsonr(
        f_ensmean_count_NWP[i_yr1 : i_yr2 + 1], obs_TC_count_NWP[i_yr1 : i_yr2 + 1]
    )
    corr_HR_NWP_list[i], p_HR_NWP_list[i] = pearsonr(
        f_ensmean_count_NWP[i_yr1 : i_yr2 + 1], obs_TY_count_NWP[i_yr1 : i_yr2 + 1]
    )

## Table

In [19]:
df = pd.DataFrame(
    [
        np.round(corr_TC_NA_list, 2),
        np.round(corr_HR_NA_list, 2),
        np.round(corr_TC_NEP_list, 2),
        np.round(corr_HR_NEP_list, 2),
        np.round(corr_TC_NWP_list, 2),
        np.round(corr_HR_NWP_list, 2),
    ],
    columns=["1940-2023", "1949-2023", "1949-1985", "1987-2023", "1970-2023"],
    index=[
        "NA TC corr",
        "NA HR corr",
        "NEP TC corr",
        "NEP HR corr",
        "NWP TC corr",
        "NWP HR corr",
    ],
)
df

,1940-2023,1949-2023,1949-1985,1987-2023,1970-2023
NA TC corr,0.45,0.45,0.32,0.61,0.45
NA HR corr,0.60,0.64,0.64,0.60,0.67
NEP TC corr,NaN,0.41,0.30,0.34,0.28
NEP HR corr,NaN,0.46,0.26,0.48,0.43
NWP TC corr,NaN,0.35,0.32,0.31,0.32
NWP HR corr,NaN,0.23,0.17,0.31,0.31


In [20]:
df = pd.DataFrame(
    [
        np.round(p_TC_NA_list, 2),
        np.round(p_HR_NA_list, 2),
        np.round(p_TC_NEP_list, 2),
        np.round(p_HR_NEP_list, 2),
        np.round(p_TC_NWP_list, 2),
        np.round(p_HR_NWP_list, 2),
    ],
    columns=["1940-2023", "1949-2023", "1949-1985", "1987-2023", "1970-2023"],
    index=[
        "NA TC p-value",
        "NA HR p-value",
        "NEP TC p-value",
        "NEP HR p-value",
        "NWP TC p-value",
        "NWP HR p-value",
    ],
)
df

,1940-2023,1949-2023,1949-1985,1987-2023,1970-2023
NA TC p-value,0.0,0.00,0.05,0.00,0.00
NA HR p-value,0.0,0.00,0.00,0.00,0.00
NEP TC p-value,NaN,0.00,0.07,0.04,0.04
NEP HR p-value,NaN,0.00,0.12,0.00,0.00
NWP TC p-value,NaN,0.00,0.05,0.06,0.02
NWP HR p-value,NaN,0.05,0.33,0.06,0.02
